# ProteinGym Phase-1: optimization-induced proxy decay (T4, self-contained)

**Question:** is a learned fitness proxy (ESM-2 zero-shot) good in bulk but **blind at the elite** that
directed evolution converges to? Metric: normalized **top-k utility** (range-restriction-free) — if you
assay the proxy's top-k, what fraction of achievable fitness do you capture? (1=optimal, 0=random).

**Why Colab:** avoids the flaky Harvard host + conda SSL. Oracle from the HuggingFace mirror; the ESM-2
proxy is computed here by **WT-marginals** (one forward pass per assay → seconds on a T4).

**Runtime → T4 GPU, then Run all (~5-10 min).** Pre-registered read: CONFIRM if bulk Spearman looks healthy
but median top-10 utility is well below 1 (proxies leave elite fitness on the table). Report whichever.


### 1. Install + GPU check


In [ ]:
!pip install -q huggingface_hub transformers pyarrow 2>/dev/null
import torch, numpy as np, pandas as pd
from scipy.stats import spearmanr
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


### 2. Download the DMS assays (oracle) from the HuggingFace mirror

The mirror stores all assays in 5 sharded Parquet files (~110MB); we load them and filter to the shortlist.


In [ ]:
import os
from huggingface_hub import list_repo_files, hf_hub_download
REPO='OATML-Markslab/ProteinGym_v1'
shards=[f for f in list_repo_files(REPO,repo_type='dataset') if f.startswith('DMS_substitutions/') and f.endswith('.parquet')]
print('parquet shards:',len(shards))
big=pd.concat([pd.read_parquet(hf_hub_download(REPO,s,repo_type='dataset')) for s in shards],ignore_index=True)
print('total rows',len(big),'| columns:',list(big.columns))
idcol=next((c for c in ['DMS_id','DMS_ID','assay_id','DMS','id'] if c in big.columns),None)
if idcol is None:
    for c in big.columns:
        if big[c].dtype==object and big[c].astype(str).str.contains('BLAT_ECOLX|GFP_AEQVI',regex=True).any(): idcol=c; break
print('assay-id column:',idcol,'| n assays total:',big[idcol].nunique())
PREFIXES=['SPG1_STRSG','GFP_AEQVI','GRB2_HUMAN','PABP_YEAST','HIS7_YEAST','BLAT_ECOLX',
          'AMIE_PSEAE','CALM1_HUMAN','UBE4B_MOUSE','P53_HUMAN']
ids=big[idcol].astype(str); assay_dfs={}
for p in PREFIXES:
    match=sorted([x for x in ids.unique() if x.startswith(p)])
    if not match: print('  no assay for',p); continue
    aid=match[0]; assay_dfs[aid]=big[ids==aid].copy(); print('  ',aid,len(assay_dfs[aid]),'variants')
print(len(assay_dfs),'assays selected')


### 3. Load ESM-2 650M (masked-LM head, for WT-marginal zero-shot)


In [ ]:
from transformers import AutoTokenizer, EsmForMaskedLM
MODEL='facebook/esm2_t33_650M_UR50D'; dev='cuda'
tok=AutoTokenizer.from_pretrained(MODEL)
model=EsmForMaskedLM.from_pretrained(MODEL).eval().half().to(dev)
AA='ACDEFGHIKLMNPQRSTVWY'; aa2id={a:tok.convert_tokens_to_ids(a) for a in AA}
print('ESM-2 650M ready; vocab ids ok:', all(v is not None and v>=0 for v in aa2id.values()))


### 4. WT-marginal proxy scores per assay (one forward pass each)


In [ ]:
def reconstruct_wt(df):
    '''WT target seq: revert one single-substitution variant.'''
    sing=df[~df['mutant'].astype(str).str.contains(':')]
    for _,r in sing.head(200).iterrows():
        m=str(r['mutant']); s=str(r['mutated_sequence'])
        try: wt,pos,mt=m[0],int(m[1:-1]),m[-1]
        except: continue
        if 0<pos<=len(s) and s[pos-1]==mt: return s[:pos-1]+wt+s[pos:]
    return str(df.iloc[0]['mutated_sequence'])
@torch.no_grad()
def wt_marginal_logprobs(wt):
    enc=tok(wt[:1022],return_tensors='pt').to(dev)
    lp=torch.log_softmax(model(**enc).logits[0].float(),dim=-1).cpu().numpy()  # (L+2, vocab)
    return lp
def score_variants(df,lp):
    out=[]
    for m in df['mutant'].astype(str):
        s=0.0; ok=True
        for sub in m.split(':'):
            try: wt,pos,mt=sub[0],int(sub[1:-1]),sub[-1]
            except: ok=False; break
            if pos>=lp.shape[0]-1 or wt not in aa2id or mt not in aa2id: ok=False; break
            s+=lp[pos,aa2id[mt]]-lp[pos,aa2id[wt]]   # <cls>=idx0 -> residue pos at token idx=pos
        out.append(s if ok else np.nan)
    return np.array(out)
assays={}
for name,df in assay_dfs.items():
    if 'DMS_score' not in df or 'mutant' not in df or 'mutated_sequence' not in df:
        print('  skip',name,'(missing cols:',[c for c in ['DMS_score','mutant','mutated_sequence'] if c not in df],')'); continue
    wt=reconstruct_wt(df); lp=wt_marginal_logprobs(wt)
    proxy=score_variants(df,lp)
    ok=~np.isnan(proxy)
    assays[name]=dict(oracle=df['DMS_score'].values[ok].astype(float),
                      proxy=proxy[ok],
                      nmut=df['mutant'].astype(str).map(lambda x:x.count(':')+1).values[ok])
    print(f'  {name}: WT len {len(wt)}, {ok.sum()}/{len(df)} scored')
print('scored',len(assays),'assays')


### 5. Decay metrics (range-restriction-free top-k utility) + verdict


In [ ]:
import json
KS=[10,25,50,100]; UFRACS=[0.01,0.02,0.05,0.10,0.25]
def topk_utility(proxy,oracle,k):
    k=min(k,len(proxy))
    if k<3: return np.nan
    floor=float(np.mean(oracle))
    pk=float(np.mean(oracle[np.argsort(-proxy)[:k]])); best=float(np.mean(oracle[np.argsort(-oracle)[:k]]))
    return (pk-floor)/(best-floor) if best-floor>1e-12 else np.nan
def ndcg(proxy,oracle,k):
    k=min(k,len(proxy)); rel=pd.Series(oracle).rank(pct=True).values
    disc=1/np.log2(np.arange(2,k+2)); order=np.argsort(-proxy)[:k]
    idcg=np.sum(np.sort(rel)[::-1][:k]*disc); return float(np.sum(rel[order]*disc)/idcg) if idcg>0 else np.nan
rows={}
for name,d in assays.items():
    p,o=d['proxy'],d['oracle']
    if len(p)<50: continue
    m=dict(n=int(len(p)),global_spearman=float(spearmanr(p,o).correlation),
           utility_at_k={k:topk_utility(p,o,k) for k in KS},
           ndcg_at_k={k:ndcg(p,o,k) for k in KS},
           utility_curve={f:topk_utility(p,o,max(3,int(len(p)*f))) for f in UFRACS})
    rows[name]=m
    print(f'[{name}] n={m["n"]:>6}  Spearman={m["global_spearman"]:+.3f}  '
          f'util@10={m["utility_at_k"][10]:+.2f}  util@100={m["utility_at_k"][100]:+.2f}')
gs=np.array([r['global_spearman'] for r in rows.values()])
u10=np.array([r['utility_at_k'][10] for r in rows.values()])
frac=float(np.mean((u10<0.7)&(gs>0.3))); conf=(frac>=0.5) or (np.nanmedian(u10)<0.7 and np.median(gs)>0.3)
print('\n================ VERDICT ================')
print(f'median global Spearman (what the field reports): {np.median(gs):+.3f}')
print(f'median top-10 utility  (what DE actually gets):  {np.nanmedian(u10):+.3f}')
print(f'assays where the standard metric oversells (util@10<0.7, global>0.3): {frac*100:.0f}%')
print('PRE-REGISTERED:', 'CONFIRM — proxies leave elite fitness on the table; flagship figure exists'
      if conf else 'WEAK — proxies already select the elite well; pivot lead')
json.dump({'assays':rows,'verdict':{'median_global_spearman':float(np.median(gs)),
          'median_util10':float(np.nanmedian(u10)),'frac_oversold':frac,'confirm':bool(conf)}},
          open('pg_decay.json','w'),indent=2)
print('\nsaved pg_decay.json — download it (Files pane) and send it back')


### 6. The decay figure


In [ ]:
import matplotlib; import matplotlib.pyplot as plt
if not rows:
    print('No assays scored — check cell 2/4 output above (id column, prefixes, or columns).')
else:
    xs=[int(f*100) for f in UFRACS]
    plt.figure(figsize=(5.4,3.3))
    for r in rows.values(): plt.plot(xs,[r['utility_curve'][f] for f in UFRACS],'-',alpha=.3,color='#1f77b4')
    mean=[np.nanmean([r['utility_curve'][f] for r in rows.values()]) for f in UFRACS]
    plt.plot(xs,mean,'o-',color='#d62728',lw=2.5,label='mean top-k utility')
    plt.axhline(1,color='.6',ls=':',label='optimal'); plt.axhline(0,color='.6',ls='--',label='random')
    plt.xscale('log'); plt.xlabel('selection budget (top-X% by proxy) — optimizer → small')
    plt.ylabel('normalized top-k utility'); plt.title('Self-defeating surrogate on ProteinGym (ESM-2)')
    plt.legend(fontsize=7.5,loc='lower right'); plt.tight_layout(); plt.savefig('pg_decay.png',dpi=170)
    print('saved pg_decay.png'); plt.show()
